In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

In [ ]:
import os
import json
import time
import re
from collections import Counter
from datetime import datetime
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import entropy
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from openai import OpenAI
from IPython.display import display, HTML


In [ ]:
# @title Import libraries and define configuration

import random
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150


In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths — resolve sensibly for both Colab (/content/...) and local (repo root)
try:
    import google.colab  # noqa: F401
    _REPO_ROOT = Path("/content")
except ImportError:
    # Local: notebook lives in repo root, so relative paths work directly
    _REPO_ROOT = Path(".")

PDF_FOLDER        = _REPO_ROOT / "pdfs"
OUTPUT_FOLDER     = _REPO_ROOT / "outputs"
CHECKPOINT_FOLDER = _REPO_ROOT / "checkpoints"
DATA_FOLDER       = _REPO_ROOT / "data"

# Ensure output directories exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)


import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

In [ ]:
# @title Load dialogue_utils — shared helper functions
# All helper functions (show_status, show_complete, show_warning,
# save_checkpoint, load_checkpoint, extract_chunks_from_pdf, extract_phrases,
# label_cluster, normalized_entropy, run_sensitivity, etc.) are imported from
# dialogue_utils.py.  The module must be on sys.path (it is fetched in the
# cell below).

try:
    import pub_dialogue.utils as du
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("dialogue_utils imported successfully")
except ImportError as _e:
    print(f"dialogue_utils not found: {_e}")
    print("Fetch it with: !wget https://raw.githubusercontent.com/mlatcl/pub-dialogue/main/dialogue_utils.py")
    raise

In [ ]:
# @title Configure API access
import os as _os

api_key = None

# 1. Try Colab secrets (when running in Google Colab)
try:
    from google.colab import userdata as _userdata
    api_key = _userdata.get('OPENAI_API_KEY')
    if api_key:
        print("API key loaded from Colab secrets")
except Exception:
    pass

# 2. Try environment variable (when running locally)
if not api_key:
    api_key = _os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("API key loaded from OPENAI_API_KEY environment variable")

# 3. Interactive prompt fallback
if not api_key:
    from getpass import getpass as _getpass
    api_key = _getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

In [ ]:
# @title Load extraction artifacts
# Loads all outputs written by 01_processing.ipynb so this notebook never
# calls the OpenAI extraction/embedding API or re-processes PDFs.

import json, numpy as np, pandas as pd
from pathlib import Path
from openpyxl import load_workbook  # noqa: F401 (ensures openpyxl available)

chunks_df    = pd.read_csv(OUTPUT_FOLDER / "paragraph_chunks.csv")
concerns_df  = pd.read_csv(OUTPUT_FOLDER / "extracted_concerns.csv")
benefits_df  = pd.read_csv(OUTPUT_FOLDER / "extracted_benefits.csv")

concern_embeddings = np.load(CHECKPOINT_FOLDER / "concern_embeddings.npy")
benefit_embeddings = np.load(CHECKPOINT_FOLDER / "benefit_embeddings.npy")

# Reconstruct concern_ids / benefit_ids if checkpointed
def _load_json(p):
    with open(p) as _f: return json.load(_f)

concern_ids = (
    _load_json(CHECKPOINT_FOLDER / "concern_ids.json")
    if (CHECKPOINT_FOLDER / "concern_ids.json").exists()
    else concerns_df["phrase"].tolist()
)
benefit_ids = (
    _load_json(CHECKPOINT_FOLDER / "benefit_ids.json")
    if (CHECKPOINT_FOLDER / "benefit_ids.json").exists()
    else benefits_df["phrase"].tolist()
)

# Rebuild metadata_lookup needed by benefit merge cell
_meta_candidates = list(OUTPUT_FOLDER.glob("*.xlsx")) + list(Path(".").glob("*.xlsx"))
if _meta_candidates:
    metadata_df = pd.read_excel(_meta_candidates[0])
    metadata_lookup = metadata_df.set_index("filename").to_dict("index")
else:
    metadata_lookup = {}
    print("WARNING: metadata xlsx not found — metadata_lookup is empty")

TECHNOLOGY_CATEGORIES = sorted(chunks_df["technology_meta"].dropna().unique().tolist())

print(f"Chunks:   {len(chunks_df):,}")
print(f"Concerns: {len(concerns_df):,}  |  embeddings: {concern_embeddings.shape}")
print(f"Benefits: {len(benefits_df):,}  |  embeddings: {benefit_embeddings.shape}")

In [ ]:
# @title Cluster concern phrase embeddings

show_status(f"Clustering into {N_CONCERN_CLUSTERS} concern groups...")

embeddings_normalized = normalize(embeddings)

kmeans = KMeans(
    n_clusters=N_CONCERN_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
concern_cluster_assignments = kmeans.fit_predict(embeddings_normalized)

centroids = kmeans.cluster_centers_
centroids_normalized = normalize(centroids)

assert len(concern_cluster_assignments) == len(concerns_df), (
    f"Cluster assignment length ({len(concern_cluster_assignments)}) does not match "
    f"concerns_df rows ({len(concerns_df)})."
)

# Add cluster assignment to concerns dataframe
concerns_df['cluster_id'] = concern_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
soft_membership = cosine_similarity(embeddings_normalized, centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_CONCERN_CLUSTERS}")
print(f"  Concern phrases: {len(concerns_df)}")
print(f"  Soft membership matrix: {soft_membership.shape}")

cluster_sizes = np.bincount(concern_cluster_assignments)
print(f"  Cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={np.median(cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "soft_membership.npy", soft_membership)
np.save(CHECKPOINT_FOLDER / "cluster_centroids.npy", centroids_normalized)

show_complete("Clustering complete")


In [ ]:
# @title Characterise clusters by technology distribution

show_status("Analysing cluster composition...")

# For each cluster, compute entropy of technology distribution
cluster_entropy = {}
cluster_tech_dist = {}

technologies = concerns_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_data = concerns_df[cluster_mask]

    if len(cluster_data) == 0:
        cluster_entropy[cluster_id] = 0
        cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    cluster_entropy[cluster_id] = entropy(tech_probs)
    cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
cluster_entropy_norm = {k: v / max_entropy for k, v in cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters = [k for k, v in cluster_entropy_norm.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters = [k for k, v in cluster_entropy_norm.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nCluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters)}")
print(f"  Ratio: {len(cross_cutting_clusters)/N_CONCERN_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(cluster_entropy_norm.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [cluster_sizes[i] for i in range(N_CONCERN_CLUSTERS)]
entropies = [cluster_entropy_norm[i] for i in range(N_CONCERN_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Cluster composition analysis complete")

# Persist entropy dicts so analysis notebooks can load them without re-running clustering
with open(OUTPUT_FOLDER / "cluster_entropy.json", "w") as _f:
    json.dump({
        "raw":  {str(k): v for k, v in cluster_entropy.items()},
        "norm": {str(k): v for k, v in cluster_entropy_norm.items()},
        "cross_cutting": cross_cutting_clusters,
    }, _f)

In [ ]:
# --- Define cross-cutting helpers for Section 6 (missing in v10-2) ---

# Use the same threshold already used in Section 3
CROSS_CUTTING_ENTROPY_THRESHOLD = CROSS_CUTTING_THRESHOLD  # already 0.5 in the notebook

# IDs of cross-cutting clusters
cross_cutting_cluster_ids = set(cross_cutting_clusters)

# Optional: map cluster_id -> human label if we have GPT labels (summary_df)
label_map = {}
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    label_map = dict(zip(summary_df["cluster_id"], summary_df["label"]))

# Provide the dict Section 6.2 expects
cross_cutting_labels = {cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids}


---
# SECTION 4: Cluster interpretation and labelling
---

In [ ]:
# @title Extract exemplar concern phrases per cluster

show_status("Extracting exemplar concerns...")

N_EXEMPLARS = 8
cluster_exemplars = {}

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_concerns = concerns_df[cluster_mask]
    cluster_embs = embeddings_normalized[cluster_mask]

    if len(cluster_concerns) == 0:
        continue

    centroid = centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for idx in top_indices:
        row = cluster_concerns.iloc[idx]
        exemplars.append({
            'concern': row['concern'],
            'technology': row['technology'],
            'year': int(row['year']) if pd.notna(row['year']) else None,
            'similarity': float(similarities[idx])
        })

    # Cluster metadata
    tech_dist = cluster_concerns['technology'].value_counts().head(3).to_dict()

    cluster_exemplars[cluster_id] = {
        'size': len(cluster_concerns),
        'entropy': normalized_entropy[cluster_id],
        'is_cross_cutting': cluster_id in cross_cutting_clusters,
        'top_technologies': tech_dist,
        'exemplars': exemplars
    }

with open(OUTPUT_FOLDER / "cluster_exemplars.json", 'w') as f:
    json.dump(cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(cluster_exemplars)} clusters")

In [ ]:
# @title Assign descriptive labels to clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

def label_cluster(cluster_id, exemplars, is_cross_cutting):
    exemplar_texts = "\n".join([f"- {ex['concern']}" for ex in exemplars[:8]])

    prompt = f"""Analyze this cluster of public concerns from UK dialogue reports.

Concern phrases in this cluster:
{exemplar_texts}

Provide:
1. SHORT LABEL (3-6 words) capturing the core concern theme.
   Use neutral, generic language; do NOT prefix the label with a specific
   technology (e.g. write "Job displacement", not "AI-driven job displacement").
2. DESCRIPTION (1-2 sentences)
3. KEY TERMS (3-5 representative words/phrases)

Return JSON only:
{{"label": "...", "description": "...", "key_terms": ["...", "..."]}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Expert qualitative researcher. Return only valid JSON."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=1000,
        )

        content = response.choices[0].message.content
        if content is None or not content.strip():
            return {'label': f'Cluster {cluster_id}', 'description': '', 'key_terms': [], 'success': False}

        content = content.strip()
        if '```' in content:
            parts = content.split('```')
            for part in parts:
                if part.startswith('json'):
                    content = part[4:].strip()
                    break
                elif part.strip().startswith('{'):
                    content = part.strip()
                    break

        result = json.loads(content)
        result['success'] = True
        return result
    except Exception as e:
        return {'label': f'Cluster {cluster_id}', 'description': '', 'key_terms': [], 'success': False, 'error': str(e)}

# Check for cached labels
labels_file = OUTPUT_FOLDER / "cluster_labels.json"

if labels_file.exists():
    print("Found cached cluster labels...")
    with open(labels_file) as f:
        cluster_labels_dict = json.load(f)

    # Verify we have labels for all clusters
    if len(cluster_labels_dict) == N_CONCERN_CLUSTERS:
        print(f"Using cached labels for {len(cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        cluster_labels_dict = None
else:
    cluster_labels_dict = None

if cluster_labels_dict is None:
    show_status(f"Generating labels for {len(cluster_exemplars)} clusters...")

    cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(cluster_exemplars.keys(), desc="Labeling"):
        data = cluster_exemplars[cluster_id]
        result = label_cluster(cluster_id, data['exemplars'], data['is_cross_cutting'])
        cluster_labels_dict[str(cluster_id)] = result

        if not result.get('success', False):
            failed_count += 1

        time.sleep(0.3)

    with open(labels_file, 'w') as f:
        json.dump(cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in cluster_labels_dict.values() if v.get('success', False))
_label_total = max(1, len(cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%}). Re-run cluster labelling or inspect failures "
        f"before continuing; downstream framing-lens analysis depends on these labels."
    )

# Create label list
cluster_labels_list = [cluster_labels_dict.get(str(i), {}).get('label', f'Cluster {i}')
                       for i in range(N_CONCERN_CLUSTERS)]

# Summary
print("\nTop 15 Clusters by Size:")
cluster_summary = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    cluster_summary.append({
        'cluster_id': cluster_id,
        'label': label,
        'size': data['size'],
        'entropy': data['entropy'],
        'type': 'Cross-cutting' if data['is_cross_cutting'] else 'Tech-specific'
    })

summary_df = pd.DataFrame(cluster_summary).sort_values('size', ascending=False)
print(summary_df[['cluster_id', 'label', 'size', 'type']].head(15).to_string(index=False))

summary_df.to_csv(OUTPUT_FOLDER / "cluster_summary.csv", index=False)


---
# SECTION 5: Framing lens analysis
---

In [ ]:
# @title Identify framing lenses

show_status("Generating framing lens suggestions...")

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_CONCERN_CLUSTERS} concern clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their concerns.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_lenses = json.loads(content)

    print("Suggested Framing Lenses:\n")
    for lens in suggested_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_lenses['framing_lenses'])} framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_lenses = {'framing_lenses': []}


In [ ]:
# @title Map concern clusters to framing lenses

FRAMING_LENS_MAPPINGS = {}
for lens in suggested_lenses['framing_lenses']:
    FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "framing_lens_mappings.json", 'w') as f:
    json.dump(FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses
cluster_to_lens = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in cluster_to_lens:
            cluster_to_lens[cluster_id] = []
        cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_CONCERN_CLUSTERS))
covered_cluster_ids = set(cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_CONCERN_CLUSTERS} concern clusters are not assigned to any "
        f"framing lens. The lens-suggestion step did not honour the 'all clusters covered' "
        f"requirement. Re-run the framing-lens identification cell or assign the missing "
        f"clusters manually before continuing."
    )

print("Framing Lens Mappings:")
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    # Count concerns in this lens
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    n_concerns = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_concerns} concerns")


In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

In [ ]:
# @title Hierarchical structure of concern clusters (dendrogram)
# Validates the framing-lens scheme by showing whether concern clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing concern-cluster dendrogram...")

# L2-normalise centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(centroids, axis=1, keepdims=True)
centroids_normed = centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and average linkage
distances = pdist(centroids_normed, metric="cosine")
Z = linkage(distances, method="average")

# Build leaf labels from existing cluster labels
cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in cluster_labels_dict.items()
}
leaf_labels = [cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(centroids_normed))]

# Build cluster->lens lookup
cluster_to_lens = {}
for lens_name, info in FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names = list(FRAMING_LENS_MAPPINGS.keys())
lens_colours = {name: cm.tab20(i / max(len(lens_names), 1))
                for i, name in enumerate(lens_names)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours.get(lens, "black"))

ax.set_xlabel("Cosine distance between concern cluster centroids")
ax.set_title("Hierarchical structure of concern clusters\n(leaf colour = LLM-assigned framing lens)")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: ARI between LLM lens assignment and dendrogram cut
from sklearn.metrics import adjusted_rand_score

n_lenses = len(FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses, criterion="maxclust")
lens_assignment = np.array([
    lens_names.index(cluster_to_lens.get(i, lens_names[0]))
    if cluster_to_lens.get(i) in lens_names else -1
    for i in range(len(centroids_normed))
])
mask = lens_assignment >= 0
ari = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of concern lenses (LLM-derived): {n_lenses}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses} groups: {ari:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Concern-cluster dendrogram complete.")

In [ ]:
# @title Cluster benefit phrase embeddings

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

show_status(f"Clustering into {N_BENEFIT_CLUSTERS} benefit groups...")

benefit_embeddings_normalized = normalize(benefit_embeddings)

benefit_kmeans = KMeans(
    n_clusters=N_BENEFIT_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
benefit_cluster_assignments = benefit_kmeans.fit_predict(benefit_embeddings_normalized)

benefit_centroids = benefit_kmeans.cluster_centers_
benefit_centroids_normalized = normalize(benefit_centroids)

assert len(benefit_cluster_assignments) == len(benefits_df), (
    f"Benefit cluster assignment length ({len(benefit_cluster_assignments)}) "
    f"does not match benefits_df rows ({len(benefits_df)})."
)

# Add cluster assignment to benefits dataframe
benefits_df["cluster_id"] = benefit_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
benefit_soft_membership = cosine_similarity(benefit_embeddings_normalized, benefit_centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_BENEFIT_CLUSTERS}")
print(f"  Benefit phrases: {len(benefits_df)}")
print(f"  Soft membership matrix: {benefit_soft_membership.shape}")

benefit_cluster_sizes = np.bincount(benefit_cluster_assignments)
print(f"  Cluster sizes: min={benefit_cluster_sizes.min()}, max={benefit_cluster_sizes.max()}, median={np.median(benefit_cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "benefit_soft_membership.npy", benefit_soft_membership)
np.save(CHECKPOINT_FOLDER / "benefit_cluster_centroids.npy", benefit_centroids_normalized)

show_complete("Benefit clustering complete")


In [ ]:
# @title Characterise benefit clusters by technology distribution

from scipy.stats import entropy as scipy_entropy

show_status("Analysing benefit cluster composition...")

benefit_cluster_entropy = {}
benefit_cluster_tech_dist = {}

technologies = benefits_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df['cluster_id'] == cluster_id
    cluster_data = benefits_df[cluster_mask]

    if len(cluster_data) == 0:
        benefit_cluster_entropy[cluster_id] = 0
        benefit_cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    benefit_cluster_entropy[cluster_id] = scipy_entropy(tech_probs)
    benefit_cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
normalized_entropy_benefits = {k: v / max_entropy for k, v in benefit_cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nBenefit Cluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters_benefits)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters_benefits)}")
print(f"  Ratio: {len(cross_cutting_clusters_benefits)/N_BENEFIT_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(normalized_entropy_benefits.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Benefit Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [int((benefits_df["cluster_id"] == i).sum()) for i in range(N_BENEFIT_CLUSTERS)]
entropies = [normalized_entropy_benefits[i] for i in range(N_BENEFIT_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Benefit cluster composition analysis complete")

# Persist benefit entropy dicts so analysis notebooks can load them without re-running clustering
with open(OUTPUT_FOLDER / "benefit_cluster_entropy.json", "w") as _f:
    json.dump({
        "raw":  {str(k): v for k, v in benefit_cluster_entropy.items()},
        "norm": {str(k): v for k, v in normalized_entropy_benefits.items()},
        "cross_cutting": cross_cutting_clusters_benefits,
    }, _f)


In [ ]:
# --- Define cross-cutting helpers for Benefits (Section 6) ---

CROSS_CUTTING_ENTROPY_THRESHOLD = globals().get("CROSS_CUTTING_THRESHOLD", 0.5)

if "cross_cutting_clusters_benefits" not in globals():
    raise NameError("cross_cutting_clusters_benefits not found. Run the benefit entropy/classification cell first.")

cross_cutting_cluster_ids_benefits = set(cross_cutting_clusters_benefits)

label_map = {}
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    label_map = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))

cross_cutting_labels_benefits = {
    cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids_benefits
}


---
# SECTION 4B: Cluster interpretation and labelling (benefits)
---

In [ ]:
# @title Extract exemplar benefit phrases per cluster

show_status("Extracting exemplar benefits...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)
N_EXEMPLARS = 8
benefit_cluster_exemplars = {}

assert "cluster_id" in benefits_df.columns, "benefits_df has no cluster_id column. Run cell 58 first."
assert benefits_df["cluster_id"].max() < N_BENEFIT_CLUSTERS, (
    f"benefits_df.cluster_id contains values >= {N_BENEFIT_CLUSTERS}. "
    "This usually means stale cluster IDs from a different clustering run; re-run cell 58."
)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df["cluster_id"] == cluster_id
    cluster_benefits = benefits_df[cluster_mask]
    cluster_embs = benefit_embeddings_normalized[cluster_mask]

    if len(cluster_benefits) == 0:
        continue

    centroid = benefit_centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for i in top_indices:
        row = cluster_benefits.iloc[i]
        exemplars.append({
            "benefit": row["benefit"],
            "technology": row["technology"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "similarity": float(similarities[i])
        })

    tech_dist = cluster_benefits["technology"].value_counts().head(3).to_dict()

    benefit_cluster_exemplars[cluster_id] = {
        "size": len(cluster_benefits),
        "entropy": normalized_entropy_benefits[cluster_id],
        "is_cross_cutting": cluster_id in cross_cutting_clusters_benefits,
        "top_technologies": tech_dist,
        "exemplars": exemplars
    }

with open(OUTPUT_FOLDER / "benefit_cluster_exemplars.json", "w") as f:
    json.dump(benefit_cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(benefit_cluster_exemplars)} clusters")


In [ ]:
# @title Assign descriptive labels to benefit clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

def label_benefit_cluster(cluster_id, exemplars, is_cross_cutting):
    """    function of the same name.
    fix for label_cluster() in cell 24."""
    exemplar_texts = "\n".join([f"- {ex['benefit']}" for ex in exemplars[:8]])

    prompt = f"""Analyze this cluster of public benefits from UK dialogue reports.

Benefit phrases in this cluster:
{exemplar_texts}

Provide:
1. SHORT LABEL (3-6 words) capturing the core benefit theme.
   Use neutral, generic language; do NOT prefix the label with a specific
   technology (e.g. write "Improved diagnosis", not "AI-driven improved diagnosis").
2. DESCRIPTION (1-2 sentences)
3. KEY TERMS (3-5 representative words/phrases)

Return JSON only:
{{"label": "...", "description": "...", "key_terms": ["...", "..."]}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Expert qualitative researcher. Return only valid JSON."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=1000,
        )
        content = response.choices[0].message.content
        if content is None or not content.strip():
            return {"label": f"Cluster {cluster_id}", "description": "", "key_terms": [], "success": False}

        content = content.strip()
        if "```" in content:
            parts = content.split("```")
            for part in parts:
                if part.startswith("json"):
                    content = part[4:].strip()
                    break
                elif part.strip().startswith("{"):
                    content = part.strip()
                    break

        result = json.loads(content)
        result["success"] = True
        return result
    except Exception as e:
        return {"label": f"Cluster {cluster_id}", "description": "", "key_terms": [], "success": False, "error": str(e)}

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Ensure exemplars dict exists
if "benefit_cluster_exemplars" not in globals():
    raise NameError("benefit_cluster_exemplars not found. Run the exemplar extraction cell first.")

# Cache for benefit labels
benefit_labels_file = OUTPUT_FOLDER / "benefit_cluster_labels.json"

benefit_cluster_labels_dict = None
if benefit_labels_file.exists():
    print("Found cached benefit cluster labels...")
    with open(benefit_labels_file) as f:
        benefit_cluster_labels_dict = json.load(f)

    if len(benefit_cluster_labels_dict) == N_BENEFIT_CLUSTERS:
        print(f"Using cached labels for {len(benefit_cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        benefit_cluster_labels_dict = None

if benefit_cluster_labels_dict is None:
    show_status(f"Generating labels for {len(benefit_cluster_exemplars)} clusters...")

    benefit_cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(benefit_cluster_exemplars.keys(), desc="Labeling"):
        data = benefit_cluster_exemplars[cluster_id]
        result = label_benefit_cluster(cluster_id, data["exemplars"], data["is_cross_cutting"])
        benefit_cluster_labels_dict[str(cluster_id)] = result

        if not result.get("success", False):
            failed_count += 1

        time.sleep(0.3)

    with open(benefit_labels_file, "w") as f:
        json.dump(benefit_cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(benefit_cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in benefit_cluster_labels_dict.values() if v.get("success", False))
_label_total = max(1, len(benefit_cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Benefit cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Benefit cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%})."
    )

# Create label list (optional downstream use)
benefit_cluster_labels_list = [
    benefit_cluster_labels_dict.get(str(i), {}).get("label", f"Cluster {i}")
    for i in range(N_BENEFIT_CLUSTERS)
]

# Summary table
print("\nTop 15 Benefit Clusters by Size:")
cluster_summary = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get("label", f"Cluster {cluster_id}")
    cluster_summary.append({
        "cluster_id": cluster_id,
        "label": label,
        "size": data["size"],
        "entropy": data["entropy"],
        "type": "Cross-cutting" if data["is_cross_cutting"] else "Tech-specific"
    })

benefit_summary_df = pd.DataFrame(cluster_summary).sort_values("size", ascending=False)
print(benefit_summary_df[["cluster_id", "label", "size", "type"]].head(15).to_string(index=False))

benefit_summary_df.to_csv(OUTPUT_FOLDER / "benefit_cluster_summary.csv", index=False)


---
# SECTION 5B: Framing lens analysis (benefits)
---

In [ ]:
# @title Identify benefit framing lenses

show_status("Generating benefit framing lens suggestions...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_BENEFIT_CLUSTERS} benefit clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their benefits.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_benefit_lenses = json.loads(content)

    print("Suggested Benefit Framing Lenses:\n")
    for lens in suggested_benefit_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_benefit_lenses['framing_lenses'])} benefit framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_benefit_lenses = {'framing_lenses': []}


In [ ]:
# @title Map benefit clusters to framing lenses

BENEFIT_FRAMING_LENS_MAPPINGS = {}
for lens in suggested_benefit_lenses['framing_lenses']:
    BENEFIT_FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "benefit_framing_lens_mappings.json", 'w') as f:
    json.dump(BENEFIT_FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses (benefit-specific)
benefit_cluster_to_lens = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in benefit_cluster_to_lens:
            benefit_cluster_to_lens[cluster_id] = []
        benefit_cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_BENEFIT_CLUSTERS))
covered_cluster_ids = set(benefit_cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered benefit cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_BENEFIT_CLUSTERS} benefit clusters are not assigned to any "
        f"framing lens. Re-run the framing-lens identification cell or assign manually."
    )

print("Benefit Framing Lens Mappings:")
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    n_benefits = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_benefits} benefits")


In [ ]:
# @title Compare results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init=10)
    _ = km_base.fit_predict(embeddings_normalized)
    baseline_centroids = km_base.cluster_centers_.copy()

# Baseline lens membership: baseline cluster -> set(lenses)
baseline_cluster_to_lenses = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

def run_for_k(k: int):
    # Re-cluster on the same (normalised) embeddings
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(embeddings_normalized)
    centroids = km.cluster_centers_.copy()

    df = concerns_df.copy()
    df['cluster_id_k'] = labels

    # --- Stable core analogue: entropy across technologies × global prevalence
    global_prev = df['cluster_id_k'].value_counts(normalize=True)
    ent = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        tech_counts = df.loc[mask, TECH_COL].value_counts()
        probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum()>0 else np.array([])
        ent[cid] = float(shannon_entropy(probs)) if len(probs) > 1 else 0.0

    stable_df = pd.DataFrame({
        'cluster_id_k': list(range(k)),
        'tech_entropy': [ent[i] for i in range(k)],
        'global_prevalence': [float(global_prev.get(i,0)) for i in range(k)]
    }).sort_values(['global_prevalence','tech_entropy'], ascending=False)

    # Plot
    plt.figure(figsize=(7,5))
    plt.scatter(stable_df['tech_entropy'], stable_df['global_prevalence'])
    plt.xlabel("Entropy across technologies")
    plt.ylabel("Share of all extracted concerns")
    plt.title(f"Stable core structure (k={k})")
    plt.tight_layout()
    plt.savefig(OUTPUT_FOLDER / f"sensitivity_stable_core_k{k}.png")
    plt.show()

    stable_df.to_csv(OUTPUT_FOLDER / f"sensitivity_stable_core_k{k}.csv", index=False)

    # --- AI fingerprint vs non-AI baseline on clusters (technology-weighted)
    techs = sorted(df[TECH_COL].dropna().unique().tolist())
    non_ai = [t for t in techs if t != 'AI']

    # Technology profiles
    prof = {}
    for t in techs:
        m = df[TECH_COL]==t
        total = m.sum()
        if total==0:
            continue
        counts = df.loc[m,'cluster_id_k'].value_counts()
        prof[t] = (counts/total).to_dict()
    prof_df = pd.DataFrame(prof).T.fillna(0)
    non_ai_avg = prof_df.loc[non_ai].mean(axis=0) if len(non_ai) else pd.Series(dtype=float)

    if 'AI' in prof_df.index and len(non_ai_avg)>0:
        diff = (prof_df.loc['AI'] - non_ai_avg).sort_values()
        top_low = diff.head(5).index.tolist()
        top_high = diff.tail(7).index.tolist()
        sel = top_low + top_high

        categories = [f"Cluster {c}" for c in sel]
        ai_vals = prof_df.loc['AI', sel].tolist()
        base_vals = non_ai_avg[sel].tolist()

        categories_loop = categories + [categories[0]]
        ai_loop = ai_vals + [ai_vals[0]]
        base_loop = base_vals + [base_vals[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
        fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI avg (tech‑weighted)'))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
            title=f"AI fingerprint vs non‑AI baseline (k={k})"
        )
        fig.write_html(OUTPUT_FOLDER / f"sensitivity_ai_fingerprint_clusters_k{k}.html")
        fig.show()

    # --- AI over time (headline): entropy / concentration of AI concern distribution by year
    ai_df = df[df[TECH_COL]=='AI'].dropna(subset=['year'])
    if len(ai_df)>0:
        yearly = []
        for yr, g in ai_df.groupby('year'):
            counts = g['cluster_id_k'].value_counts(normalize=True)
            ent_y = float(shannon_entropy(counts.values)) if len(counts)>1 else 0.0
            top10 = float(counts.nlargest(min(10,len(counts))).sum())
            hhi = float((counts.values**2).sum())
            yearly.append({'year': int(yr), 'entropy': ent_y, 'top10_share': top10, 'hhi': hhi})
        yd = pd.DataFrame(yearly).sort_values('year')
        yd.to_csv(OUTPUT_FOLDER / f"sensitivity_ai_time_metrics_k{k}.csv", index=False)

        plt.figure(figsize=(8,4))
        plt.plot(yd['year'], yd['entropy'], marker='o')
        plt.title(f"AI discourse diversity over time (entropy, k={k})")
        plt.xlabel("Year")
        plt.ylabel("Entropy of cluster distribution")
        plt.tight_layout()
        plt.savefig(OUTPUT_FOLDER / f"sensitivity_ai_entropy_over_time_k{k}.png")
        plt.show()

    # --- AI vs non-AI by framing lens (mapped to baseline lens vocabulary)
    # Map new clusters to nearest baseline centroid, then inherit baseline lens memberships
    sim = cosine_similarity(centroids, baseline_centroids)
    nearest = sim.argmax(axis=1)  # baseline cluster id for each new cluster
    new_cluster_to_lenses = {}
    for new_cid in range(k):
        base_cid = int(nearest[new_cid])
        new_cluster_to_lenses[new_cid] = baseline_cluster_to_lenses.get(base_cid, set())

    # Lens salience per tech
    lens_names = list(FRAMING_LENS_MAPPINGS.keys())
    lens_by_tech = {}
    for t in techs:
        m = df[TECH_COL]==t
        total = m.sum()
        if total==0:
            continue
        lens_by_tech[t] = {ln:0.0 for ln in lens_names}
        # count concerns by lens via cluster mapping
        for cid, cnt in df.loc[m,'cluster_id_k'].value_counts().items():
            for ln in new_cluster_to_lenses.get(int(cid), set()):
                lens_by_tech[t][ln] += cnt
        # normalise
        for ln in lens_names:
            lens_by_tech[t][ln] = lens_by_tech[t][ln] / total

    lens_df = pd.DataFrame(lens_by_tech).T.fillna(0)
    lens_df.to_csv(OUTPUT_FOLDER / f"sensitivity_lens_salience_by_tech_k{k}.csv")

    if 'AI' in lens_df.index and len(non_ai)>0:
        non_ai_avg_lens = lens_df.loc[non_ai].mean(axis=0)
        cats = list(non_ai_avg_lens.index)
        ai_vals = lens_df.loc['AI', cats].tolist()
        base_vals = non_ai_avg_lens[cats].tolist()

        cats_loop = cats + [cats[0]]
        ai_loop = ai_vals + [ai_vals[0]]
        base_loop = base_vals + [base_vals[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(r=ai_loop, theta=cats_loop, fill='toself', name='AI'))
        fig.add_trace(go.Scatterpolar(r=base_loop, theta=cats_loop, fill='toself', name='Non‑AI avg (tech‑weighted)'))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
            title=f"AI framing profile vs non‑AI baseline (mapped lenses, k={k})"
        )
        fig.write_html(OUTPUT_FOLDER / f"sensitivity_ai_framing_radar_k{k}.html")
        fig.show()

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    run_for_k(k)

show_complete("Sensitivity runs complete (see outputs folder)")


In [ ]:
# @title Compare benefit results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for benefit cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_benefit_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init=10)
    _ = km_base.fit_predict(benefit_embeddings_normalized)
    baseline_benefit_centroids = km_base.cluster_centers_.copy()

baseline_cluster_to_lenses = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

def run_for_k(k: int):
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(benefit_embeddings_normalized)
    centroids = km.cluster_centers_.copy()

    df = benefits_df.copy()
    df['cluster_id_k'] = labels

    # Stable core analogue: entropy across technologies x global prevalence
    global_prev = df['cluster_id_k'].value_counts(normalize=True)
    ent = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        tech_counts = df.loc[mask, TECH_COL].value_counts()
        probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
        ent[cid] = float(shannon_entropy(probs)) if len(probs) > 1 else 0.0

    # Normalize entropy to [0, 1]
    n_techs = df[TECH_COL].nunique()
    max_ent = np.log(n_techs) if n_techs > 1 else 1.0
    norm_ent = {cid: e / max_ent for cid, e in ent.items()}

    cross_cutting = [cid for cid, e in norm_ent.items() if e >= 0.5]

    # Map this k's clusters to baseline (k=75) cluster centroids by cosine
    sims = cosine_similarity(centroids, baseline_benefit_centroids)
    best_baseline = sims.argmax(axis=1)  # for each new cluster, the closest baseline cluster

    # Inherit baseline lens membership
    cluster_to_lenses_k = {
        cid: baseline_cluster_to_lenses.get(int(best_baseline[cid]), set())
        for cid in range(k)
    }

    # Lens prevalence in this k
    lens_prev = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        n = mask.sum()
        for lens in cluster_to_lenses_k[cid]:
            lens_prev[lens] = lens_prev.get(lens, 0) + n

    total = sum(lens_prev.values())
    if total > 0:
        lens_prev = {l: v / total for l, v in lens_prev.items()}

    return {
        'k': k,
        'n_cross_cutting': len(cross_cutting),
        'pct_cross_cutting': len(cross_cutting) / k,
        'lens_prev': lens_prev,
        'top_lenses': sorted(lens_prev.items(), key=lambda x: -x[1])[:5],
    }

results = {}
for k in ks:
    print(f"\n--- k = {k} ---")
    results[k] = run_for_k(k)
    r = results[k]
    print(f"  Cross-cutting clusters: {r['n_cross_cutting']} ({r['pct_cross_cutting']:.0%})")
    print("  Top lenses:")
    for lens, prev in r['top_lenses']:
        print(f"    {lens}: {prev:.1%}")

# Build comparison table
all_lenses = set()
for r in results.values():
    all_lenses.update(r['lens_prev'].keys())

comp_rows = []
for lens in sorted(all_lenses):
    row = {'framing_lens': lens}
    for k in ks:
        row[f'k={k}'] = results[k]['lens_prev'].get(lens, 0)
    comp_rows.append(row)

comp_df = pd.DataFrame(comp_rows).sort_values(f'k={BASELINE_K}', ascending=False)

print("\n--- Lens prevalence by cluster count ---")
display(comp_df)

comp_df.to_csv(OUTPUT_FOLDER / "benefit_sensitivity_lens_prevalence.csv", index=False)

summary_rows = [{
    'k': k,
    'n_cross_cutting': results[k]['n_cross_cutting'],
    'pct_cross_cutting': results[k]['pct_cross_cutting'],
} for k in ks]
pd.DataFrame(summary_rows).to_csv(OUTPUT_FOLDER / "benefit_sensitivity_summary.csv", index=False)

show_complete("Benefit cluster-count sensitivity analysis complete")


In [ ]:
# @title Artifact manifest — verify all expected outputs were written
# This cell asserts that every artifact expected by the analysis notebooks
# exists on disk.  Run it at the end of 01_processing to confirm a complete run.

from pathlib import Path
_out  = OUTPUT_FOLDER
_ckpt = CHECKPOINT_FOLDER

_EXPECTED_OUTPUT = [
    "paragraph_chunks.csv",
    "paragraph_chunks_per_document.csv",
    "extracted_concerns.csv",
    "extracted_benefits.csv",
    "cluster_labels.json",
    "cluster_summary.csv",
    "cluster_exemplars.json",
    "cluster_entropy.json",
    "framing_lens_mappings.json",
    "benefit_cluster_labels.json",
    "benefit_cluster_summary.csv",
    "benefit_cluster_exemplars.json",
    "benefit_cluster_entropy.json",
    "benefit_framing_lens_mappings.json",
]

_EXPECTED_CHECKPOINT = [
    "concern_embeddings.npy",
    "concern_ids.json",
    "cluster_centroids.npy",
    "benefit_embeddings.npy",
    "benefit_ids.json",
    "benefit_cluster_centroids.npy",
]

_missing = []
for _name in _EXPECTED_OUTPUT:
    _p = _out / _name
    if _p.exists():
        print(f"  OK   {_name}")
    else:
        print(f"  MISS {_name}")
        _missing.append(str(_p))

for _name in _EXPECTED_CHECKPOINT:
    _p = _ckpt / _name
    if _p.exists():
        print(f"  OK   checkpoints/{_name}")
    else:
        print(f"  MISS checkpoints/{_name}")
        _missing.append(str(_p))

if _missing:
    raise RuntimeError(
        f"\n{len(_missing)} artifact(s) missing — check earlier cells:\n"
        + "\n".join(f"  {m}" for m in _missing)
    )
print(f"\nAll {len(_EXPECTED_OUTPUT) + len(_EXPECTED_CHECKPOINT)} artifacts present.")